# 1. Setup and Data Preparation
Prepare the runtime environment and load the preprocessed dataset for training.

## Mount Google Drive
Mount Google Drive so the notebook can access the shared dataset archive and any saved outputs.

In [1]:
import sys

# 若在 Colab 環境執行，才掛載 Google Drive
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("非 Colab 環境 (本機執行)，跳過掛載 Google Drive。")

非 Colab 環境 (本機執行)，跳過掛載 Google Drive。


## Extract the Dataset Archive
Unzip the preprocessed dataset into the Colab runtime so file access is fast during training and evaluation.

In [2]:
import os
import sys

# 僅在 Colab 環境執行解壓縮
if 'google.colab' in sys.modules:
    # 將雲端硬碟的 .zip 檔解壓縮到 Colab 本地空間 (讀取速度最快)
    zip_path = "/content/drive/MyDrive/Hand_gestures/dataset_v3_processed_detected.zip" 
    extract_path = "/content/processed_hagrid_small"

    if os.path.exists(zip_path) and not os.path.exists(extract_path):
        print("解壓縮資料集至 Colab 空間中！ (這會大幅提升訓練速度)...")
        # 加上 -o 參數，如果檔案存在就直接蓋過去，不要跳出詢問視窗卡死
        !unzip -q -o "{zip_path}" -d /content/
        print("解壓縮完成！")
    elif os.path.exists(extract_path):
        print("資料集已解壓縮過了，跳過解壓縮。")
    else:
        print(f"請確認您的雲端硬碟根目錄真的有 {zip_path} 這個檔案！")
else:
    print("本機環境，跳過解壓縮。請確保資料集已經放在正確的資料夾中。")

本機環境，跳過解壓縮。請確保資料集已經放在正確的資料夾中。


## Load the Manifest Table
Read the CSV manifest and normalize stored file paths so they point to the extracted Colab dataset location.

In [ ]:
import pandas as pd
from pathlib import Path

# Colab 測試時的預設解壓縮路徑
DATA_CSV = Path("/content/labels.csv")

if not DATA_CSV.exists():
    # 本機測試時的備用路徑 (Windows / Mac) D:\Hand_Gesture\data\processed_hagrid_small_detect
    DATA_CSV = Path("dataset_v3_processed_detected/labels.csv")
    if not DATA_CSV.exists():
        # 如果還是找不到，可以修改這個絕對路徑 (例如 Windows D 槽)
        DATA_CSV = Path("D:/Hand_Gesture/data/dataset_v2_processed_detected/labels.csv")

df = pd.read_csv(DATA_CSV)

# === 修復路徑：支援各平台 ===
# 無論 CSV 內存的是 Mac 還是 Windows 的絕對路徑
# 統一拿相對結構 "crops/檔名" 組合上目前 CSV 實際所在的資料夾
base_dir = DATA_CSV.parent

def fix_path_cross_platform(path_str):
    if pd.isna(path_str): return path_str
    # 解決 Unix 下讀取 Windows 反斜線會被當成檔名的問題
    path_str = path_str.replace("\\", "/")
    p_path = Path(path_str)
    # 取出 "crops/檔名" 或 "landmarks/檔名"，與真正的 base_dir 結合
    return str(base_dir / p_path.parent.name / p_path.name)

df["crop_path"] = df["crop_path"].apply(fix_path_cross_platform)
df["landmark_path"] = df["landmark_path"].apply(fix_path_cross_platform)

print("using:", DATA_CSV)
print(df.head())
print("total:", len(df))
print(df["label_name"].value_counts())
print(df["label"].value_counts().sort_index())

using: D:\Hand_Gesture\data\dataset_v2_processed_detected\labels.csv
                                                 idx original_class_folder  \
0          holy_533b48d8-0985-4920-bc0e-e8c87bb36ef0                  holy   
1         peace_50907f0a-555f-4d3b-8780-3bafa6b07449                 peace   
2  little_finger_56a147ef-1e6c-4a56-9050-e863a5c5...         little_finger   
3         point_13990b00-42cf-418a-8d0e-9c72ef58d1b1                 point   
4         peace_80c42f7c-fa52-4ee7-9fb7-df62012a875a                 peace   

   label label_name                                          crop_path  \
0      0        N_A  D:\Hand_Gesture\data\dataset_v2_processed_dete...   
1      0        N_A  D:\Hand_Gesture\data\dataset_v2_processed_dete...   
2      0        N_A  D:\Hand_Gesture\data\dataset_v2_processed_dete...   
3      0        N_A  D:\Hand_Gesture\data\dataset_v2_processed_dete...   
4      0        N_A  D:\Hand_Gesture\data\dataset_v2_processed_dete...   

                 

## Split: Training / Validation / Testing
Create stratified train, validation, and test splits so each class keeps a similar distribution across all subsets.

In [4]:
from sklearn.model_selection import train_test_split

SEED = 42

# 如果需要測試用少量樣本，可以取消註解底下這行覆寫：
# df = pd.read_csv("D:/Hand_Gesture/data/processed_sample/labels.csv")

# Split train 70%, temp 30%
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

# Split temp 30% into val 15% and test 15%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

# Reset index for all DataFrames
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("using:", DATA_CSV)
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

print("\nTrain class count:")
print(train_df["label_name"].value_counts())

print("\nVal class count:")
print(val_df["label_name"].value_counts())

print("\nTest class count:")
print(test_df["label_name"].value_counts())

using: D:\Hand_Gesture\data\dataset_v2_processed_detected\labels.csv
train: 69997
val: 14999
test: 15000

Train class count:
label_name
N_A     37287
palm     6964
ok       6858
fist     6497
like     6238
one      6153
Name: count, dtype: int64

Val class count:
label_name
N_A     7990
palm    1492
ok      1469
fist    1392
like    1337
one     1319
Name: count, dtype: int64

Test class count:
label_name
N_A     7990
palm    1492
ok      1470
fist    1393
like    1337
one     1318
Name: count, dtype: int64


# 2. Landmark-only Training
This notebook trains a small classifier that uses only the 21 hand landmarks.
The cropped image is still loaded only to keep the padding/resize coordinate transform consistent with the fusion model.

## Runtime Device and Basic Imports
Set up PyTorch, the training device, and common utilities used below.

In [5]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.notebook import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("torch version:", torch.__version__)
print("device:", device)
if torch.cuda.is_available():
    print("torch cuda version:", torch.version.cuda)
    print("device count:", torch.cuda.device_count())

torch version: 2.8.0+cu129
device: cuda
torch cuda version: 12.9
device count: 1


## Import Padding Resize and Augmentor
Use the same padding/resize function and the provided `UltimateDataAugmentor`.
For landmark-only training, image-only augmentations such as blur and color jitter are disabled.

In [6]:
# If running in Colab, allow imports from your Google Drive project folder.
if 'google.colab' in sys.modules:
    drive_project_dir = "/content/drive/MyDrive/Hand_gestures"
    if drive_project_dir not in sys.path:
        sys.path.append(drive_project_dir)

try:
    from padding_resize import pad_to_square_and_resize
    from augmentor_fixed import UltimateDataAugmentor
except ModuleNotFoundError as e:
    print(f"Cannot import required module: {e}")
    if 'google.colab' in sys.modules:
        print("Please upload padding_resize.py and augmentor_fixed.py to /content or Google Drive/Hand_gestures.")
    else:
        print("Please put padding_resize.py and augmentor_fixed.py in the same folder as this notebook.")
    raise e


landmark_train_augmentor = UltimateDataAugmentor(
    p_blur=0.0,
    p_color=0.0,
    p_flip=0.5,
    max_rotate=15.0,
    max_scale=0.05,
    max_translate=0.03
)

## Landmark-only Dataset
The dataset returns only `(landmarks, label)`.
The crop image is loaded only so `pad_to_square_and_resize()` can transform landmark coordinates consistently.

In [7]:
class LandmarkOnlyDataset(Dataset):
    def __init__(self, df, is_train=False, augmentor=None):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.augmentor = augmentor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["crop_path"]).convert("RGB")
        landmarks = np.load(row["landmark_path"]).astype(np.float32)
        label = int(row["label"])

        # Keep the same coordinate transform as the fusion model.
        img, landmarks = pad_to_square_and_resize(
            img,
            landmarks,
            target_size=224
        )

        # Apply synchronized geometric augmentation only during training.
        if self.is_train and self.augmentor is not None:
            img, landmarks = self.augmentor(img, landmarks)

        # Wrist-centered normalization, same idea as the fusion model landmark branch.
        wrist = landmarks[0, :]
        rel_landmarks = landmarks - wrist

        max_dist = np.max(np.abs(rel_landmarks))
        if max_dist > 0:
            norm_landmarks = rel_landmarks / max_dist
        else:
            norm_landmarks = rel_landmarks

        landmarks = torch.tensor(norm_landmarks, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return landmarks, label

## Create Landmark DataLoaders
Build train, validation, and test loaders.

In [ ]:
import cv2
cv2.setNumThreads(0)
BATCH_SIZE = 64

train_landmark_dataset = LandmarkOnlyDataset(
    train_df,
    is_train=True,
    augmentor=landmark_train_augmentor
)

val_landmark_dataset = LandmarkOnlyDataset(
    val_df,
    is_train=False,
    augmentor=None
)

test_landmark_dataset = LandmarkOnlyDataset(
    test_df,
    is_train=False,
    augmentor=None
)

NUM_WORKERS = 2

train_landmark_loader = DataLoader(
    train_landmark_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

val_landmark_loader = DataLoader(
    val_landmark_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

test_landmark_loader = DataLoader(
    test_landmark_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

landmarks, labels = next(iter(train_landmark_loader))
print("landmarks:", landmarks.shape)
print("labels:", labels.shape)

## Landmark-only Model
A small MLP that takes flattened 21 × 2 landmarks and predicts the 6 gesture classes.

In [9]:
class LandmarkOnlyModel(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, num_classes=6):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, num_classes)
        )

    def forward(self, landmarks):
        if landmarks.dim() == 3:
            landmarks = landmarks.view(landmarks.size(0), -1)
        return self.mlp(landmarks)


model = LandmarkOnlyModel(
    input_dim=42,
    hidden_dim=128,
    num_classes=6
).to(device)

print(model)

LandmarkOnlyModel(
  (mlp): Sequential(
    (0): Linear(in_features=42, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=64, out_features=6, bias=True)
  )
)


## Training Utilities
Training and validation functions for landmark-only mini-batches.

In [10]:
def train_one_epoch_landmark(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for landmarks, labels in tqdm(loader, desc="Training", leave=False):
        landmarks = landmarks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(landmarks)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_landmark(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for landmarks, labels in tqdm(loader, desc="Validation", leave=False):
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(landmarks)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

## Train and Save the Landmark-only Model
Train the model and save the best validation checkpoint.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

if 'google.colab' in sys.modules:
    save_dir = Path("/content/drive/MyDrive/Hand_gestures")
elif platform.system() == "Windows":
    save_dir = Path("D:/Hand_Gesture/model")
else:
    save_dir = Path("./model")

save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "landmark_only_aug_best.pth"

best_val_acc = 0.0
num_epochs = 10

history = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch_landmark(
        model,
        train_landmark_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = evaluate_landmark(
        model,
        val_landmark_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best landmark-only model: val_acc={best_val_acc:.4f}")

print("\nBest validation accuracy:", best_val_acc)
print("Saved to:", save_path)

history_df = pd.DataFrame(history)
display(history_df.tail())


Epoch 1/30


Training:   0%|          | 0/1094 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Detailed Evaluation
Compute overall accuracy, target-class accuracy, N/A false trigger rate, and a confusion matrix.

In [ ]:
class_names = ["N_A", "fist", "like", "ok", "one", "palm"]


def evaluate_detailed_landmark(model, loader, criterion=None, threshold=None):
    model.eval()

    total_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_confs = []

    with torch.no_grad():
        for landmarks, labels in tqdm(loader, desc="Detailed evaluation", leave=False):
            landmarks = landmarks.to(device)
            labels = labels.to(device)

            logits = model(landmarks)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)

            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            if threshold is not None:
                preds = torch.where(
                    confs < threshold,
                    torch.zeros_like(preds),
                    preds
                )

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

            total += labels.size(0)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_confs = np.array(all_confs)

    accuracy = (all_labels == all_preds).mean()

    target_mask = all_labels != 0
    na_mask = all_labels == 0

    target_accuracy = (
        (all_labels[target_mask] == all_preds[target_mask]).mean()
        if target_mask.sum() > 0 else 0.0
    )

    na_false_trigger_rate = (
        (all_preds[na_mask] != 0).mean()
        if na_mask.sum() > 0 else 0.0
    )

    metrics = {
        "loss": total_loss / total if criterion is not None else None,
        "accuracy": accuracy,
        "target_accuracy": target_accuracy,
        "na_false_trigger_rate": na_false_trigger_rate,
        "total": total,
    }

    result_df = pd.DataFrame({
        "true": all_labels,
        "pred": all_preds,
        "confidence": all_confs,
        "true_name": [class_names[i] for i in all_labels],
        "pred_name": [class_names[i] for i in all_preds],
    })

    cm = pd.crosstab(
        result_df["true_name"],
        result_df["pred_name"],
        rownames=["True"],
        colnames=["Pred"],
        dropna=False
    )

    cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)

    return metrics, cm, result_df

## Load Best Checkpoint and Evaluate
Evaluate the saved best model on validation and test sets.

In [ ]:
best_model = LandmarkOnlyModel(
    input_dim=42,
    hidden_dim=128,
    num_classes=6
).to(device)

best_model.load_state_dict(
    torch.load(save_path, map_location=device)
)
best_model.eval()

print("Validation:")
val_metrics, val_cm, val_result_df = evaluate_detailed_landmark(
    best_model,
    val_landmark_loader,
    criterion=criterion,
    threshold=None
)
print(val_metrics)
display(val_cm)

print("\nTest:")
test_metrics, test_cm, test_result_df = evaluate_detailed_landmark(
    best_model,
    test_landmark_loader,
    criterion=criterion,
    threshold=None
)
print(test_metrics)
display(test_cm)
display(test_result_df.head())

## Save fp16 Version
Save a smaller fp16 state_dict for inference/submission experiments.

In [ ]:
def save_model_fp16(model, save_path):
    state_dict = model.state_dict()
    state_dict_fp16 = {}

    for k, v in state_dict.items():
        if torch.is_floating_point(v):
            state_dict_fp16[k] = v.half()
        else:
            state_dict_fp16[k] = v

    torch.save(state_dict_fp16, save_path)


fp16_path = save_dir / "landmark_only_aug_best_fp16.pth"
save_model_fp16(best_model, fp16_path)

print("fp32 size:", f"{save_path.stat().st_size / 1024:.2f} KB")
print("fp16 size:", f"{fp16_path.stat().st_size / 1024:.2f} KB")
print("saved:", fp16_path)

## Optional: Save Dynamic Quantized Version
This quantizes the Linear layers for CPU inference experiments.
It is optional; compare accuracy and file size before using it.

In [ ]:
quant_path = save_dir / "landmark_only_aug_best_dynamic_int8.pth"

model_q = LandmarkOnlyModel(
    input_dim=42,
    hidden_dim=128,
    num_classes=6
).cpu()

model_q.load_state_dict(torch.load(save_path, map_location="cpu"))
model_q.eval()

model_q = torch.ao.quantization.quantize_dynamic(
    model_q,
    {nn.Linear},
    dtype=torch.qint8
)

torch.save(model_q.state_dict(), quant_path)

print("fp32 size:", f"{save_path.stat().st_size / 1024:.2f} KB")
print("fp16 size:", f"{fp16_path.stat().st_size / 1024:.2f} KB")
print("int8 size:", f"{quant_path.stat().st_size / 1024:.2f} KB")
print("saved:", quant_path)

## Optional: Evaluate Dynamic Quantized Version on CPU
Dynamic quantized models generally run on CPU.

In [ ]:
def evaluate_detailed_landmark_cpu(model, loader, criterion=None, threshold=None):
    model.eval()

    total_loss = 0.0
    total = 0

    all_labels = []
    all_preds = []
    all_confs = []

    with torch.no_grad():
        for landmarks, labels in tqdm(loader, desc="CPU detailed evaluation", leave=False):
            landmarks = landmarks.cpu()
            labels = labels.cpu()

            logits = model(landmarks)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)

            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            if threshold is not None:
                preds = torch.where(
                    confs < threshold,
                    torch.zeros_like(preds),
                    preds
                )

            all_labels.extend(labels.numpy())
            all_preds.extend(preds.numpy())
            all_confs.extend(confs.numpy())

            total += labels.size(0)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_confs = np.array(all_confs)

    accuracy = (all_labels == all_preds).mean()

    target_mask = all_labels != 0
    na_mask = all_labels == 0

    target_accuracy = (
        (all_labels[target_mask] == all_preds[target_mask]).mean()
        if target_mask.sum() > 0 else 0.0
    )

    na_false_trigger_rate = (
        (all_preds[na_mask] != 0).mean()
        if na_mask.sum() > 0 else 0.0
    )

    metrics = {
        "loss": total_loss / total if criterion is not None else None,
        "accuracy": accuracy,
        "target_accuracy": target_accuracy,
        "na_false_trigger_rate": na_false_trigger_rate,
        "total": total,
    }

    result_df = pd.DataFrame({
        "true": all_labels,
        "pred": all_preds,
        "confidence": all_confs,
        "true_name": [class_names[i] for i in all_labels],
        "pred_name": [class_names[i] for i in all_preds],
    })

    cm = pd.crosstab(
        result_df["true_name"],
        result_df["pred_name"],
        rownames=["True"],
        colnames=["Pred"],
        dropna=False
    )
    cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)

    return metrics, cm, result_df


model_q_eval = LandmarkOnlyModel(
    input_dim=42,
    hidden_dim=128,
    num_classes=6
).cpu()

model_q_eval = torch.ao.quantization.quantize_dynamic(
    model_q_eval,
    {nn.Linear},
    dtype=torch.qint8
)

model_q_eval.load_state_dict(torch.load(quant_path, map_location="cpu"))
model_q_eval.eval()

q_metrics, q_cm, q_result_df = evaluate_detailed_landmark_cpu(
    model_q_eval,
    test_landmark_loader,
    criterion=nn.CrossEntropyLoss(),
    threshold=None
)

print(q_metrics)
display(q_cm)